In [14]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [4]:
class ConvNextBlock(nn.Module):
  def __init__(self, dim, layer_scale):
    super(ConvNextBlock, self).__init__()
    self.depthwiseConv = nn.Conv2d(dim, dim, 7, padding=3, groups=dim)
    self.norm = nn.LayerNorm(dim)
    self.conv_1 = nn.Conv2d(dim, dim*4, kernel_size=1)
    self.activation = nn.GELU()
    self.conv_2 = nn.Conv2d(dim*4, dim, kernel_size=1)
    self.layer_scale = nn.Parameter(torch.ones(dim,1,1)*layer_scale)

  def forward(self, x):
    original_input = x

    print("Input:", x.shape)

    x = self.depthwiseConv(x)
    print("After Depthwise Conv:", x.shape)

    x = x.permute(0, 2, 3, 1)
    print("After permute:", x.shape)

    x = self.norm(x)
    print("After LayerNorm:", x.shape)

    x = x.permute(0, 3, 1, 2)
    print("After re-permute:", x.shape)

    x = self.conv_1(x)
    print("After conv_1:", x.shape)

    x = self.activation(x)
    print("After activation:", x.shape)
    x = self.conv_2(x)
    print("After conv_2:", x.shape)

    x = x + original_input

    return x

In [5]:
# Initialize block
block = ConvNextBlock(dim=192, layer_scale=1e-6)

# Create dummy input
x = torch.randn(1, 192, 56, 56)

# Forward pass
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input: torch.Size([1, 192, 56, 56])
After Depthwise Conv: torch.Size([1, 192, 56, 56])
After permute: torch.Size([1, 56, 56, 192])
After LayerNorm: torch.Size([1, 56, 56, 192])
After re-permute: torch.Size([1, 192, 56, 56])
After conv_1: torch.Size([1, 768, 56, 56])
After activation: torch.Size([1, 768, 56, 56])
After conv_2: torch.Size([1, 192, 56, 56])
Input shape: torch.Size([1, 192, 56, 56])
Output shape: torch.Size([1, 192, 56, 56])


In [6]:
class PatchifyStemBlock(nn.Module):
  def __init__(self, input=3, output=96):
    super(PatchifyStemBlock,self).__init__()
    self.conv = nn.Conv2d(input, output, kernel_size=4, stride=4)
    self.norm = nn.LayerNorm(output)

  def forward(self, x):
    x = self.conv(x)

    x = x.permute(0, 2, 3, 1)
    x = self.norm(x)
    x = x.permute(0, 3, 1, 2)

    return x

In [7]:
class DownsamplingBlock(nn.Module):
  def __init__(self, input=3, output=96):
    super(DownsamplingBlock,self).__init__()
    self.norm = nn.LayerNorm(output)
    self.conv = nn.Conv2d(input, output, kernel_size=2, stride=2)

  def forward(self, x):
    x = x.permute(0, 2, 3, 1)
    x = self.norm(x)
    x = x.permute(0, 3, 1, 2)

    x = self.conv(x)
    return x

In [8]:
for i in range(len([3,3,9,3])):
  cur_dim = 96*(2**i)
  print(f"{i}: {cur_dim}")

0: 96
1: 192
2: 384
3: 768


In [20]:
class ConvNext(nn.Module):
  def __init__(self, num_classes=30,
               layer_scale=1e-6,
               dim = 96,
               num_blocks = [3, 3, 9, 3],
               ):
    super(ConvNext,self).__init__()

    self.num_classes = num_classes
    self.network_length = len(num_blocks)
    self.dim = dim
    self.num_blocks = num_blocks
    self.input_stem = PatchifyStemBlock()

    self.downsample_layers = nn.ModuleList()
    self.stages = nn.ModuleList()

    self.dims = []

    for i in range(self.network_length):
      cur_dim = dim*(2**i)
      self.dims.append(cur_dim)

      stage = nn.Sequential(*[
          ConvNextBlock(dim=cur_dim,layer_scale=layer_scale)
          for j in range(num_blocks[i])
        ])

      self.stages.append(stage)

      self.downsample_layers.append(
          DownsamplingBlock(cur_dim, cur_dim*2)
      )

    self.head = nn.Linear(self.dims[-1], self.num_classes)
    self.final_norm = nn.LayerNorm(self.dims[-1])

  def create_network(self):
    layers = []
    layers.append(self.input_stem)

    for i, convnext_block in enumerate(self.stages):
      layers.append(convnext_block)

      if i < self.network_length -1:
        layers.append(self.downsample_layers[i])

    self.layers = nn.Sequential(*layers)

  def forward(self, x):
      x = self.layers(x)

      # https://discuss.pytorch.org/t/global-average-pooling-in-pytorch/6721/9
      x = nn.functional.adaptive_avg_pool2d(x, (1, 1))
      x = torch.flatten(x, 1)
      x = self.final_norm(x)
      x = self.head(x)

      return x

In [21]:
conv_network = ConvNext()

In [22]:
conv_network.create_network()

In [23]:
print(conv_network.layers)

Sequential(
  (0): PatchifyStemBlock(
    (conv): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (1): Sequential(
    (0): ConvNextBlock(
      (depthwiseConv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
      (conv_1): Conv2d(96, 384, kernel_size=(1, 1), stride=(1, 1))
      (activation): GELU(approximate='none')
      (conv_2): Conv2d(384, 96, kernel_size=(1, 1), stride=(1, 1))
    )
    (1): ConvNextBlock(
      (depthwiseConv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True, bias=True)
      (conv_1): Conv2d(96, 384, kernel_size=(1, 1), stride=(1, 1))
      (activation): GELU(approximate='none')
      (conv_2): Conv2d(384, 96, kernel_size=(1, 1), stride=(1, 1))
    )
    (2): ConvNextBlock(
    

In [ ]:
"""
TO DO:
- EfficientNet Ideas:
-- Compound Scaling
-- use SILU instead of GELU
-- add squeeze-and-excitation optimization
-- stochastic depth

"""